# Middleware


Middleware provides a way to more tightly control what happens inside the agent. Middleware is useful for the following:
- Tracking agent behavior with logging, analytics and debugging.
- Transforming prompts, tool selection and output formatting.
- Adding retries, fallback and early termination logic.
- Applying rate limits


In [5]:
from langchain_ollama import ChatOllama
from langchain.agents import create_agent

llm = ChatOllama(
    model='qwen2.5:3b',
)

agent = create_agent(
    model= llm,
    system_prompt="You are a helpful assistant"
)



# Summarization middleware

Automatically summarize conversation history when approaching token limits, preserving recent messages while compressing older context. Summarization is useful for the following:
   - Long-running conversation that exeed context windows.
   - Multi-turn dialogues with extensive history
   - Application where preserving full conversation context matters.

In [7]:
from langchain.agents.middleware import SummarizationMiddleware
from langchain_core.messages import HumanMessage, SystemMessage

### Messagebased summarization
agent = create_agent(
    model=llm,
    middleware=[
        SummarizationMiddleware(
            model=llm,
            trigger=("messages", 10),
            keep=("messages",4)
        )
    ]
)

In [9]:
### RUn with thread id
config = {'configurable':{'thread_id':'test-1'}}

In [10]:
# Alternative test data

questions=[
    'What is 2+2?',
    'what is 10*5?',
    'what is 100/4?',
    'what is 15-7?',
    'what is 3*3?',
    'what is 4*4?'
]

for q in questions:
    response = agent.invoke({"messages":[HumanMessage(content=q)]}, config)
    print(f"messages: {response}")
    print(f"Messages: {len(response['messages'])}")

messages: {'messages': [HumanMessage(content='What is 2+2?', additional_kwargs={}, response_metadata={}, id='5b904423-bdbc-4b48-8fa6-8dfd1fdb3a45'), AIMessage(content='2 + 2 equals 4.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-26T10:31:02.931171Z', 'done': True, 'done_reason': 'stop', 'total_duration': 1548200625, 'load_duration': 1304559459, 'prompt_eval_count': 36, 'prompt_eval_duration': 80339000, 'eval_count': 9, 'eval_duration': 161747000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a177c3-7804-7719-8c74-4706a8e7e351-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 36, 'output_tokens': 9, 'total_tokens': 45})]}
Messages: 2
messages: {'messages': [HumanMessage(content='what is 10*5?', additional_kwargs={}, response_metadata={}, id='9d40ca8e-ebc2-44e3-8f4c-235c4565e910'), AIMessage(content='10*5 equals 50.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b

# Human loop middleware


In [11]:
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware

def read_email_tool(email_id: str) -> str:
    """Mock function to read an email by it's id"""
    return f"Email content for ID: {email_id}"

def send_email_tool(recipient: str, subject: str, body: str) -> str:
    """Mock function to send an email"""
    return f"Email sent to {recipient} with subject '{subject}'"


In [13]:
agent = create_agent(
    model=llm,
    tools=[read_email_tool, send_email_tool],
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "send_email_tool":{
                    "allowed_decisions":["approve", "edit", "reject"]
                },
                "read_email_tool": False
            }
        )
    ]
)

In [19]:
config = {'configurable': {'thread_id': 'test-approve'}}
# step 1: Request

result = agent.invoke(
    {'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'")]},
    config=config
)
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='a2fd78bb-d942-4957-a81e-db11b3787df6'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-26T11:03:15.311986Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2254506291, 'load_duration': 1301453083, 'prompt_eval_count': 222, 'prompt_eval_duration': 324848000, 'eval_count': 37, 'eval_duration': 620485000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a177e0-f1a0-7047-9c0a-a6b5f1a1f40a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you'}, 'id': '40ee4d6c-1f85-40e3-a53b-a027d9497472', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 222, 'output_tokens': 37, 'total_tokens': 259}),
  HumanMessage(content="Send email to john

In [20]:
from langgraph.types import Command

if "__interrupt__" in result:
    print("⏸️ Paused! Approving...")

    result = agent.invoke(
        Command(
            resume={
                'decisions': [
                    {'type': 'approve'}
                ]
            }
        ),
        config= config
    )

    print(f'✅ Result: {result['messages'][-1].content}')

⏸️ Paused! Approving...
✅ Result: Email has been sent to john@test.com with the subject 'Hello'.


In [21]:
result

{'messages': [HumanMessage(content="Send email to john@test.com with subject 'Hello' and body 'How are you'", additional_kwargs={}, response_metadata={}, id='a2fd78bb-d942-4957-a81e-db11b3787df6'),
  AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:3b', 'created_at': '2026-10-26T11:03:15.311986Z', 'done': True, 'done_reason': 'stop', 'total_duration': 2254506291, 'load_duration': 1301453083, 'prompt_eval_count': 222, 'prompt_eval_duration': 324848000, 'eval_count': 37, 'eval_duration': 620485000, 'logprobs': None, 'model_name': 'qwen2.5:3b', 'model_provider': 'ollama'}, id='lc_run--01a177e0-f1a0-7047-9c0a-a6b5f1a1f40a-0', tool_calls=[{'name': 'send_email_tool', 'args': {'recipient': 'john@test.com', 'subject': 'Hello', 'body': 'How are you'}, 'id': '40ee4d6c-1f85-40e3-a53b-a027d9497472', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 222, 'output_tokens': 37, 'total_tokens': 259}),
  HumanMessage(content="Send email to john